In [40]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[06/09/26 14:45:50] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=530539;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=693750;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [41]:
os.listdir('data/08_reporting/01_comunicaciones')

['Base de Gestión Recupero_Reincorporación UNIS_20.04.2026.xlsx']

In [42]:
df_activos = catalog.load('unis_activos_calendario')
df_bajas = catalog.load('unis_bajas_calendario_academico')

[06/09/26 14:45:51] INFO     Loading data from unis_activos_calendario (ParquetDataset)...     ]8;id=499798;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=656392;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

                    INFO     Loading data from unis_bajas_calendario_academico                 ]8;id=646040;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=534033;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

In [43]:
df_bajas.shape

(642, 41)

In [44]:
df_recuperos = pd.read_excel('data/08_reporting/01_comunicaciones/Base de Gestión Recupero_Reincorporación UNIS_20.04.2026.xlsx')

In [45]:
df_recuperos.info()

<class 'pandas.DataFrame'>
RangeIndex: 1213 entries, 0 to 1212
Data columns (total 28 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   Número Identificación                  1212 non-null   object        
 1   ID Inconcert                           1212 non-null   float64       
 2   Etapa                                  79 non-null     str           
 3   ID. Banner                             1212 non-null   float64       
 4   Nombres                                1212 non-null   str           
 5   Apellidos                              1212 non-null   str           
 6   Nivel Formación                        1212 non-null   str           
 7   Código del Programa                    1212 non-null   str           
 8   Programa                               1212 non-null   str           
 9   Correo personal                        1212 non-null   str           
 10 

In [46]:
df_bajas.info()

<class 'pandas.DataFrame'>
RangeIndex: 642 entries, 0 to 641
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_inconcert           615 non-null    float64       
 1   identificacion         641 non-null    float64       
 2   correo_2               642 non-null    str           
 3   nombres                642 non-null    str           
 4   fecha_registro         642 non-null    datetime64[ns]
 5   cohorte                642 non-null    datetime64[ns]
 6   fecha_ingreso          642 non-null    datetime64[ns]
 7   usuario_institucional  642 non-null    str           
 8   nivel                  642 non-null    str           
 9   programa               642 non-null    str           
 10  estado                 642 non-null    str           
 11  tipo_baja              642 non-null    str           
 12  motivo_baja            642 non-null    str           
 13  submotivo_baja  

In [47]:
df_bajas.columns


Index(['id_inconcert', 'identificacion', 'correo_2', 'nombres',
       'fecha_registro', 'cohorte', 'fecha_ingreso', 'usuario_institucional',
       'nivel', 'programa', 'estado', 'tipo_baja', 'motivo_baja',
       'submotivo_baja', 'comentarios', 'fecha_de_baja_t', 'fecha_de_baja_d',
       'fecha_de_reingreso', 'fecha_de_grado', 'exito_estudiantil', 'alianza',
       'etapa_studen_journey', 'di', 'gi', 'fecha_baja', 'max_semana_teorica',
       'periodo_raw', 'periodo_inicial', 'periodo_actual', 'sesion',
       'fecha_inicio', 'fecha_fin', 'shifted_fecha_inicio', 'semana',
       'semana_acumulada', 'month', 'mes_academico', 'anio_gregoriano',
       'mes_gregoriano', 'student_journey', 'tipo'],
      dtype='str')

In [48]:
df_bajas['fecha_baja'] = df_bajas['fecha_de_baja_t']
df_bajas.loc[df_bajas['fecha_baja'].isna(), 'fecha_baja'] = df_bajas.loc[df_bajas['fecha_baja'].isna(), 'fecha_de_baja_d']


In [49]:
df_recuperos.shape

(1213, 28)

In [50]:
df_recuperos_v2 = pd.merge(df_recuperos,
         df_bajas.loc[:, ['id_inconcert', 'fecha_baja']],
        left_on='ID Inconcert', 
        right_on='id_inconcert', how='left', indicator=True)

In [53]:
df_recuperos_v2['_merge'] == 'both'
df_recuperos_v2_export = df_recuperos_v2[df_recuperos_v2['_merge'] == 'both'].drop(columns=['id_inconcert', '_merge'])
df_recuperos_v2_export.to_excel('data/08_reporting/01_comunicaciones/Base de Gestión Recupero_Reincorporación UNIS_09.06.2026.xlsx', index=False)

In [52]:
df_recuperos_v2_export.shape

(572, 29)

In [37]:
df_recuperos_v2_export['fecha_baja'].value_counts()


fecha_baja
2025-06-26    61
2025-11-20    41
2025-11-17    38
2026-03-20    19
2025-01-13    12
              ..
2026-02-17     1
2026-01-14     1
2025-01-23     1
2025-04-01     1
2026-02-16     1
Name: count, Length: 186, dtype: int64

In [12]:
df_inconcert_ventas = catalog.load("unis_ventas_id_inconcert") 

[06/09/26 14:25:53] INFO     Loading data from unis_ventas_id_inconcert                        ]8;id=853801;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=495645;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (GoogleSheetsDataset)...                                                              

In [14]:

df_inconcert_ventas = df_inconcert_ventas.dropna(subset=['Identificacion'], how='any')
mask_central = df_inconcert_ventas['Universidad'] == 'UNIS'
df_inconcert_ventas_unis = df_inconcert_ventas[mask_central].copy()
# Eliminar si el movil tiene el prefijo +57 al principio del número
df_inconcert_ventas_unis['Movil'] = df_inconcert_ventas_unis['Movil'].str.replace(r'^\+57', '', regex=True)

In [15]:
df_inconcert_ventas_unis

,Universidad,Nombre,Apellido,Email,Movil,Numerodecelular2,TipoPrograma,Programa,Matricula,IdLead,CohorteVenta,Identificacion,FechaPago
28936,UNIS,CARLOS JULIO ALEJANDRO,NOVA MIRANDA,novamiranda1620@hotmail.com,55271926,NaN,Maestría,Maestría en Docencia Universitaria,650002299.0,54346,2026-07-07,1695518731801,2026-06-03
28937,UNIS,JOSÉ ANGEL,RAMÍREZ MONTEALEGRE,jarmontealegre@gmail.com,23662581,NaN,Maestría,Maestría en Docencia Universitaria,650002296.0,54319,2026-07-07,2550051560101,2026-06-02
28938,UNIS,KARLA ALEJANDRA,CELIS LÓPEZ,celisale2011@gmail.com,39903961,NaN,Maestría,Maestría en Neuropsicología Clínica,650002301.0,54316,2026-07-07,1689594880101,2026-06-04
28939,UNIS,MARIO ROLANDO,TREJO MILIÁN,mario89trejo@gmail.com,56303343,NaN,Maestría,Maestría en Docencia Universitaria,650002294.0,54083,2026-07-07,1917695330101,2026-06-02
28940,UNIS,CARLOS NOÉ,HERNÁNDEZ,carlosnoegt@yahoo.com,41396595,NaN,Maestría,Maestría en Docencia Universitaria,650002286.0,53956,2026-07-07,2431750861001,2026-05-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...
30849,UNIS,ZULEYCKA ALEJANDRA,ESCOBAR FUENTES,zuleyckaescobar@gmail.com,58375219,NaN,Maestría,Maestría en Neuropsicología Clínica,650000616.0,122,2025-01-14,3002187450101,2025-01-06
30850,UNIS,MÓNICA DEL ROCÍO,URREA HERRERA,monica.urrea26@gmail.com,50141809,NaN,Maestría,Maestría en Neuropsicología Clínica,650000459.0,114,2025-01-14,1799603930101,2024-11-14
30851,UNIS,BRENDA SUCELY,HERNANDEZ GARCIA DE ARANA,brendashg15@gmail.com,42402795,NaN,Maestría,Maestría en Neuropsicología Clínica,650000844.0,112,2025-05-13,2655385720101,2025-04-01
30852,UNIS,JULIO MARIANO,ARRIOLA DONIS,psicojulioarriola@gmail.com,42121777,NaN,Maestría,Maestría en Neuropsicología Clínica,650000000.0,110,2024-08-27,2517079111001,2024-06-05
